# How many games to rank a Kaggriculture agent

Kaggriculture is agent versus agent, and the ladder ranks you on many matches. This notebook asks a practical
question that follows from a measured fact: because the games are close, how many of them does it take before a
win-rate difference is real rather than noise?

First the fact. On the sampled episodes the median win is by only a few percent of the winner's money, and a
large share of games are decided inside a couple of percent. As the ladder matures those margins shrink further,
which tracks the field catching up to the top. Close games mean one match tells you very little. So the notebook
turns that into a tool: with a standard proportion argument it computes the number of games needed to separate
an agent from a coin flip at each edge size, which is the number to trust before reading anything into a rank.

The episode outcomes and the daily ladder come from the companion dataset, so this is reproducible from public
data. Every number is computed live, the sample size is stated, and none of it is a leaderboard claim.

## 0. Config and the data loader

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
mpl.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 120,
                     "font.size": 10.5, "axes.grid": True, "grid.alpha": 0.2, "axes.unicode_minus": False})
INK="#1b1b2f"; BL="#2f6df6"; GD="#e8a63c"; HL="#d1495b"; GR="#8a8a99"; GRN="#2a9d8f"

def find(fn):
    for c in ["/kaggle/input/kaggriculture-ladder-meta",
              "/kaggle/input/kaggriculture-ladder-meta/kaggriculture-ladder-meta"]:
        p = os.path.join(c, fn)
        if os.path.exists(p): return p
    hits = glob.glob("/kaggle/input/**/" + fn, recursive=True)
    if hits: return hits[0]
    loc = os.path.join("D:/kaggle/ds_kaggri_meta", fn)
    return loc if os.path.exists(loc) else None
print("episode_results:", find("episode_results.csv"), "| daily_ladder:", find("daily_ladder.csv"))

## 1. The sampled games

Load the per-episode outcomes and keep the games where both agents finished cleanly. The gate prints the sample
size, which is small and stated up front, and the reward range so the margins are read in context.

In [ ]:
ep = pd.read_csv(find("episode_results.csv")) if find("episode_results.csv") else pd.DataFrame()
lad = pd.read_csv(find("daily_ladder.csv")) if find("daily_ladder.csv") else pd.DataFrame()
READY = len(ep) > 0
if READY:
    v = ep[ep["both_done"] == True].copy() if "both_done" in ep.columns else ep.copy()
    v["winner_reward"] = v[["reward_a","reward_b"]].max(axis=1)
    v["margin_pct"] = 100 * v["margin"] / v["winner_reward"]
    print("episodes:", len(ep), "| complete games:", len(v), "| dates:", sorted(v["date"].unique().tolist()))
    print("reward range:", int(v["winner_reward"].min()), "-", int(v["winner_reward"].max()), "| ties:", int((ep['winner']=='tie').sum()))
    print("GATE: complete games n =", len(v))
else:
    print("companion dataset not attached; add busyaprime/kaggriculture-ladder-meta and it renders")

## 2. The games are close

The margin as a percent of the winner's reward, across the complete games. The dashed line is the median. A
large share of games sit inside a few percent, which is the whole reason a single match is a weak signal. The
printed bootstrap interval is the uncertainty on the median at this sample size.

In [ ]:
if READY:
    m = v["margin_pct"].values
    fig, ax = plt.subplots(figsize=(9.5,4.4))
    ax.hist(m, bins=30, color=BL, alpha=0.9)
    med = float(np.median(m)); ax.axvline(med, color=HL, lw=2.5, label=f"median {med:.1f}%")
    ax.set_xlabel("winning margin (% of winner reward)"); ax.set_ylabel("games"); ax.legend()
    ax.set_title("Most Kaggriculture games are won by a thin margin", color=INK, loc="left", fontweight="bold", fontsize=11)
    plt.tight_layout(); plt.show()
    rng = np.random.default_rng(0)
    boot = [np.median(rng.choice(m, len(m), replace=True)) for _ in range(5000)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    for thr in [1,2,5,10]:
        print(f"games decided by < {thr}% of winner reward: {(m<thr).mean()*100:.1f}%")
    print(f"median margin {med:.2f}% (95% bootstrap CI {lo:.2f}% to {hi:.2f}%) on n={len(m)} games")

## 3. The margin shrinks as the ladder converges

Two views of the same convergence. On the left, the median winning margin by day. On the right, from the daily
ladder, the gap between the top and median score. Both fall over time: as the field catches up to the top, wins
get closer and ranking gets harder.

In [ ]:
if READY:
    fig, (a1,a2) = plt.subplots(1,2, figsize=(12,4.3))
    by = v.groupby("date")["margin_pct"].median()
    a1.plot(range(len(by)), by.values, marker="o", color=BL, lw=2)
    a1.set_xticks(range(len(by))); a1.set_xticklabels(by.index, rotation=25, ha="right", fontsize=8)
    a1.set_ylabel("median margin (% of winner)"); a1.set_title("Winning margin by day", color=INK, loc="left", fontweight="bold")
    if len(lad) and "top_median_gap" in lad.columns:
        a2.plot(range(len(lad)), lad["top_median_gap"].values, marker="o", color=GD, lw=2)
        a2.set_xticks(range(0,len(lad),max(1,len(lad)//6)))
        a2.set_xticklabels([lad["date"].iloc[i] for i in range(0,len(lad),max(1,len(lad)//6))], rotation=25, ha="right", fontsize=8)
        a2.set_ylabel("top minus median score"); a2.set_title("Ladder top-to-median gap", color=INK, loc="left", fontweight="bold")
    plt.tight_layout(); plt.show()
    print("median margin by day:", {d: round(float(x),2) for d,x in by.items()})
    if len(lad): print("ladder top-median gap first -> last:", round(float(lad['top_median_gap'].iloc[0]),1), "->", round(float(lad['top_median_gap'].iloc[-1]),1))

## 4. The tool: games needed to beat a coin flip

Here is the payoff. Model each game as a Bernoulli trial and ask how many games make a 95% confidence interval
on the win rate exclude 0.5. For an agent whose true win rate is p, the games needed is about
`1.96^2 * p(1-p) / (p-0.5)^2`. A small edge needs hundreds of games; a large edge needs a few dozen. Read off
your suspected edge and you get the number of matches to trust before believing a rank.

In [ ]:
if READY:
    ps = np.arange(0.52, 0.76, 0.01)
    need = (1.96**2) * ps*(1-ps) / (ps-0.5)**2
    fig, ax = plt.subplots(figsize=(9.5,4.6))
    ax.plot((ps*100), need, marker="o", color=GRN, lw=2)
    ax.set_yscale("log"); ax.set_xlabel("true win rate of the stronger agent (%)")
    ax.set_ylabel("games needed (log) to exclude 50-50")
    for pv in [0.55,0.60,0.70]:
        nn=(1.96**2)*pv*(1-pv)/(pv-0.5)**2
        ax.annotate(f"{pv:.0%} -> {nn:.0f}", xy=(pv*100,nn), fontsize=9, color=INK,
                    xytext=(pv*100+0.4, nn*1.4))
    ax.set_title("A thin edge takes many games to prove", color=INK, loc="left", fontweight="bold", fontsize=11)
    plt.tight_layout(); plt.show()
    for pv in [0.55,0.60,0.65,0.70]:
        nn=(1.96**2)*pv*(1-pv)/(pv-0.5)**2
        print(f"win rate {pv:.0%}: about {nn:.0f} games to be 95% sure it beats a coin flip")

## What this shows, and how to use it

Kaggriculture games are close. On the sampled complete games the median win is a few percent of the winner's
money and a large share land inside a couple of percent, with a bootstrap interval that keeps that conclusion
even at this sample size. The margins shrink as the ladder matures, in step with the top-to-median score gap
closing, so ranking gets harder over time not easier. The consequence is concrete: a single match is a weak
signal, and separating an agent from a coin flip takes on the order of a few hundred games for a small edge and
a few dozen for a large one. Use the curve as a rule for how many matches to run before trusting a rank
difference. The sample is small and these are the sampled episodes, not the full ladder; the games-needed
argument assumes independent matches. All values are computed live. No leaderboard position is claimed.

Data: https://www.kaggle.com/datasets/busyaprime/kaggriculture-ladder-meta
Companion: https://www.kaggle.com/code/busyaprime/what-actually-wins-on-the-kaggriculture-ladder